# Lab 06 — Robustness to Real Sample Definition

### Graded Lab 3 · Student edition

**Course:** MSc Astrostatistics for Observational Cosmology **Format:** 4-hour  
in-class practical + extended graded analysis **Data:** published Pantheon+ / SH0ES  
standardized magnitudes and STAT+SYS covariance **Submission:** one executed notebook  
plus the required HTML/PDF export **Assessment:** graded extension contributes **5% of  
the course grade**

> ## Scientific question
> **How stable is the inferred cosmology under defensible changes to the
> observed sample definition?**

This lab deliberately keeps the statistical model fixed and changes the **observed  
sample definition**. The purpose is to diagnose robustness, not to search for a subset  
that gives a preferred cosmological result.

## Scope and non-negotiable distinction

This is a **real-data sensitivity analysis**.

It is **not** a formal reconstruction of the Pantheon+ survey selection function.

The Week 6 lecture derived

$$
p(D\mid\theta,S=1)
=
\frac{
p(S=1\mid D,\theta)\,p(D\mid\theta)
}{
P(S=1\mid\theta)
}.
$$

Lab 06 does not estimate either selection term in this expression. Therefore:

$$
\boxed{
\text{sample-cut sensitivity}
\neq
\text{selection-function correction}.
}
$$

Do not claim that a cut removes Malmquist bias or makes the sample unbiased.

## What remains fixed and what changes

Across every compulsory variant, keep fixed:

- the flat matter–$\Lambda$ cosmological model;
- $H_0=70\ {\rm km\,s^{-1}\,Mpc^{-1}}$ as a magnitude zero-point convention;
- uniform priors
$0.10<\Omega_m<0.60$ and $-20.2<M_B<-18.2$ mag;
- the Pantheon+ **STAT+SYS covariance**;
- the same grid resolution;
- the same posterior-summary convention.

Only the sample mask changes.

This design is essential. If the covariance model, priors and sample definition all  
change together, the source of a posterior shift becomes ambiguous.

## Intended learning outcomes and practical timeline

After this exercise you should be able to:

1. define a reproducible baseline Pantheon+ teaching sample;
2. subset a covariance matrix without breaking row alignment;
3. repeat the same posterior calculation under fixed sample perturbations;
4. quantify absolute and descriptive normalized parameter shifts;
5. diagnose whether a large shift accompanies changes in redshift, precision
or survey composition;
6. distinguish leverage and influence from invalid data;
7. distinguish robustness analysis from formal selection correction;
8. write a conclusion proportional to what the analysis actually tests.

**Suggested pacing**

- 0–20 min: five-line statement and Week 6 recall;
- 20–50 min: load data and reproduce the Lab 05 baseline;
- 50–110 min: implement fixed sample variants;
- 110–145 min: build the common robustness table;
- 145–165 min: make robustness plots;
- 165–210 min: diagnose the largest shift;
- 210–240 min: checkpoint and graded-extension briefing.

# A. Five-line scientific statement — complete before calculations

Write 5–10 sentences in your own words.

1. **Estimand:** What posterior quantity is primary? Which nuisance parameter
must also be inferred?
2. **Observed data:** Which Pantheon+ columns and covariance product enter?
3. **Generative assumptions:** How do $(\Omega_m,M_B)$ generate the expected
corrected apparent magnitudes?
4. **Uncertainty / selection:** What covariance is used? What baseline sample
is conditioned on? What is deliberately *not* corrected?
5. **Validation / falsification test:** What result would make you distrust
the implementation before interpreting robustness?

### Your response

[Replace this line with your own scientific statement.]

## A1. The primary estimand

The **primary estimand** in this lab is the marginalized posterior for

$$
\Omega_m.
$$

The nuisance magnitude $M_B$ must still be inferred and reported because sample  
changes can also alter the fitted zero point.

For each sample variant $k$, report

$$
p(\Omega_m,M_B\mid D_k)
$$

and the 16th, 50th and 84th percentiles of each marginal posterior.

MCMC is **not required** in this lab. Lab 05 already assessed MCMC. Here we use the  
deterministic fine-grid posterior so that differences between sample definitions are  
not contaminated by Monte Carlo variation.

# B. Load the published data and full covariance

Use the same official Pantheon+ / SH0ES release as Labs 03–05.

The compulsory covariance in this lab is

`Pantheon+SH0ES_STAT+SYS.cov`.

Repeated light-curve rows are retained. Their dependence is represented by the  
published covariance matrix. Do not collapse rows by `CID`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from scipy.linalg import cho_factor, cho_solve
from scipy.special import logsumexp
from numpy.polynomial.legendre import leggauss

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)

In [ ]:
BASE = (
    "https://raw.githubusercontent.com/PantheonPlusSH0ES/"
    "DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/"
)

URL_DATA = BASE + "Pantheon%2BSH0ES.dat"
URL_FULL = BASE + "Pantheon%2BSH0ES_STAT%2BSYS.cov"

CACHE = Path("data_cache/pantheon_plus")
CACHE.mkdir(parents=True, exist_ok=True)

def download_if_needed(url, path, timeout=240):
    path = Path(path)
    if path.is_file() and path.stat().st_size > 1000:
        return path

    print("Downloading:", path.name)

    response = requests.get(
        url,
        timeout=timeout,
    )
    response.raise_for_status()
    path.write_bytes(response.content)

    return path

def load_table():
    path = download_if_needed(
        URL_DATA,
        CACHE / "Pantheon+SH0ES.dat",
    )
    return pd.read_csv(
        path,
        sep=r"\s+",
    )

def load_covariance():
    path = download_if_needed(
        URL_FULL,
        CACHE / "Pantheon+SH0ES_STAT+SYS.cov",
    )

    values = np.fromfile(
        path,
        sep=" ",
    )

    n = int(values[0])

    if values.size != 1 + n * n:
        raise ValueError(
            "Unexpected covariance file dimensions."
        )

    return values[1:].reshape(
        (n, n)
    )

sn = load_table()
C_all = load_covariance()

assert len(sn) == C_all.shape[0] == C_all.shape[1]

print("Published rows:", len(sn))
print("Unique CID:", sn["CID"].nunique())

## B1. Baseline teaching sample

The compulsory baseline is exactly the non-calibrator Hubble-flow teaching selection  
used in Lab 05:

$$
z_{\rm HD}>0.01,
$$

finite `zHD` and `m_b_corr`, and

$$
{\tt IS\_CALIBRATOR}=0.
$$

This is a **teaching selection**, not the official full Pantheon+ cosmology  
likelihood.

In [ ]:
def baseline_mask(table):
    z = table["zHD"].to_numpy(float)
    m = table["m_b_corr"].to_numpy(float)
    calibrator = table["IS_CALIBRATOR"].to_numpy(int)

    return (
        np.isfinite(z)
        & np.isfinite(m)
        & (z > 0.01)
        & (calibrator == 0)
    )

def subset_with_covariance(
    table,
    covariance,
    mask,
):
    idx = np.flatnonzero(mask)

    selected = (
        table.iloc[idx]
        .reset_index(drop=True)
        .copy()
    )

    selected_covariance = covariance[
        np.ix_(idx, idx)
    ]

    assert (
        len(selected)
        == selected_covariance.shape[0]
        == selected_covariance.shape[1]
    )

    return (
        selected,
        selected_covariance,
        idx,
    )

mask0 = baseline_mask(sn)

data0, C0, idx0 = subset_with_covariance(
    sn,
    C_all,
    mask0,
)

print("Baseline rows:", len(data0))
print("Baseline unique CID:", data0["CID"].nunique())
print(
    "Baseline z range:",
    (
        float(data0["zHD"].min()),
        float(data0["zHD"].max()),
    ),
)

### Covariance row-order rule

Every sample perturbation must subset **both** the data and covariance with the same  
ordered indices.

For selected indices $I$,

$$
\boxed{
C_{\rm subset}
=
C[I,I].
}
$$

Never sort or filter the data independently of the covariance.

## B2. Survey labels used in this lab

The Pantheon+ release maps `IDSURVEY=1` to SDSS.

The compulsory leave-one-survey-out test therefore removes **SDSS** for every student.  
This makes the graded perturbation comparable across submissions.

You may explore another survey as an optional extension, but the required result uses  
`IDSURVEY != 1`.

In [ ]:
SURVEY_NAMES = {
    1: "SDSS",
    4: "SNLS",
    5: "CSP",
    10: "DES",
    15: "PS1MD",
    18: "CNIa0.02",
    50: "LOWZ/JRK07",
    51: "LOSS1",
    56: "SOUSA",
    57: "LOSS2",
    61: "CFA1",
    62: "CFA2",
    63: "CFA3S",
    64: "CFA3K",
    65: "CFA4p2",
    66: "CFA4p3",
    100: "HST",
    101: "SNAP",
    106: "CANDELS",
    150: "FOUND",
}

survey_counts = (
    data0["IDSURVEY"]
    .value_counts()
    .rename_axis("IDSURVEY")
    .reset_index(name="N")
)

survey_counts["survey"] = survey_counts[
    "IDSURVEY"
].map(SURVEY_NAMES)

display(
    survey_counts[
        ["IDSURVEY", "survey", "N"]
    ]
)

# C. Fixed cosmological model and posterior engine

The teaching model is

$$
\mathbf m^{\rm corr}
=
\boldsymbol\mu(\Omega_m,H_0)
+
M_B\mathbf1
+
\boldsymbol\epsilon,
\qquad
\boldsymbol\epsilon
\sim
\mathcal N(\mathbf0,C).
$$

The full-covariance quadratic form is

$$
\chi^2
=
\mathbf r^T
C^{-1}
\mathbf r.
$$

As in Lab 05, define

$$
\mathbf y
=
\mathbf m^{\rm corr}
-
\boldsymbol\mu(\Omega_m,H_0),
$$

and

$$
a=\mathbf y^TC^{-1}\mathbf y,
\qquad
b=\mathbf1^TC^{-1}\mathbf y,
\qquad
d=\mathbf1^TC^{-1}\mathbf1.
$$

Then

$$
\chi^2
=
\left(
a-\frac{b^2}{d}
\right)
+
d
\left(
M_B-\frac{b}{d}
\right)^2.
$$

In [ ]:
H0_CONVENTION = 70.0
OMEGA_BOUNDS = (0.10, 0.60)
MB_BOUNDS = (-20.2, -18.2)

OMEGA_GRID = np.linspace(
    *OMEGA_BOUNDS,
    201,
)

# The Lab 05 refined spacing is retained for every variant.
MB_GRID = np.linspace(
    *MB_BOUNDS,
    2821,
)

C_LIGHT_KMS = 299792.458
GL_X, GL_W = leggauss(48)

def mu_flat_lcdm(
    redshift,
    omega_m,
    H0=H0_CONVENTION,
):
    redshift = np.asarray(
        redshift,
        dtype=float,
    )

    zp = (
        redshift[..., None]
        * (GL_X + 1.0)
        / 2.0
    )

    E = np.sqrt(
        omega_m * (1.0 + zp) ** 3
        + (1.0 - omega_m)
    )

    integral = (
        redshift
        / 2.0
        * np.sum(
            GL_W / E,
            axis=-1,
        )
    )

    d_l_mpc = (
        (1.0 + redshift)
        * C_LIGHT_KMS
        / H0
        * integral
    )

    return (
        5.0 * np.log10(d_l_mpc)
        + 25.0
    )

def quantiles(
    grid,
    masses,
):
    masses = np.asarray(
        masses,
        float,
    )

    cdf = np.cumsum(
        masses / masses.sum()
    )

    return np.interp(
        [0.16, 0.50, 0.84],
        cdf,
        grid,
    )

In [ ]:
def fit_variant(
    data,
    covariance,
):
    z = data["zHD"].to_numpy(float)
    m = data["m_b_corr"].to_numpy(float)

    mu_grid = np.vstack([
        mu_flat_lcdm(
            z,
            om,
        )
        for om in OMEGA_GRID
    ])

    ygrid = (
        m[None, :]
        - mu_grid
    )

    factor = cho_factor(
        covariance,
        lower=True,
        check_finite=False,
    )

    ones = np.ones(
        len(data)
    )

    cinv_one = cho_solve(
        factor,
        ones,
        check_finite=False,
    )

    d = float(
        ones @ cinv_one
    )

    solved = cho_solve(
        factor,
        ygrid.T,
        check_finite=False,
    )

    a = np.einsum(
        "ij,ji->i",
        ygrid,
        solved,
    )

    b = ygrid @ cinv_one

    mbhat = b / d

    chi2prof = (
        a
        - b * b / d
    )

    chi2 = (
        chi2prof[:, None]
        + d
        * (
            MB_GRID[None, :]
            - mbhat[:, None]
        ) ** 2
    )

    logp = (
        -0.5
        * (
            chi2
            - np.min(chi2)
        )
    )

    logp -= logsumexp(
        logp
    )

    P = np.exp(logp)

    assert np.isfinite(P).all()
    assert np.isclose(
        P.sum(),
        1.0,
        atol=1e-10,
    )

    p_omega = P.sum(axis=1)
    p_mb = P.sum(axis=0)

    q_omega = quantiles(
        OMEGA_GRID,
        p_omega,
    )

    q_mb = quantiles(
        MB_GRID,
        p_mb,
    )

    edge_mass = float(
        P[0, :].sum()
        + P[-1, :].sum()
        + P[:, 0].sum()
        + P[:, -1].sum()
    )

    return {
        "P": P,
        "p_omega": p_omega,
        "p_mb": p_mb,
        "q_omega": q_omega,
        "q_mb": q_mb,
        "mbhat": mbhat,
        "chi2prof": chi2prof,
        "d": d,
        "edge_mass": edge_mass,
    }

# D. Checkpoint 1 — reproduce the Lab 05 baseline

Before changing the sample, reproduce the full-covariance baseline.

Your result should agree with your Lab 05 fine-grid result to numerical grid  
precision.

For orientation, the course benchmark is approximately

$$
\Omega_m \simeq 0.331,
\qquad
M_B\simeq -19.352\ {\rm mag}.
$$

If your result is substantially different, **stop and diagnose the implementation  
before continuing**.

In [ ]:
baseline_fit = fit_variant(
    data0,
    C0,
)

baseline_om = baseline_fit[
    "q_omega"
]

baseline_mb = baseline_fit[
    "q_mb"
]

print(
    "Baseline Omega_m "
    "[16, 50, 84]:",
    baseline_om,
)

print(
    "Baseline M_B "
    "[16, 50, 84]:",
    baseline_mb,
)

print(
    "Posterior edge mass:",
    baseline_fit[
        "edge_mass"
    ],
)

assert baseline_fit[
    "edge_mass"
] < 1e-6

### Checkpoint question

Does your baseline reproduce Lab 05? If yes, record the numerical differences between  
the two notebooks.

If no, do not reinterpret the disagreement scientifically. First inspect:

- covariance file;
- row selection;
- covariance indexing;
- cosmological model;
- prior bounds;
- grid resolution.

# E. Prespecified robustness variants

Every student must use the same compulsory variants.

| Variant | Rule | Main diagnostic purpose |
|---|---|---|
| Baseline | full teaching sample | reference |
| $z\le0.70$ | remove highest-redshift tail | high-$z$ leverage |
| Remove top 10% $\sigma_i$ | $\sigma_i=\sqrt{C_{ii}}$ | precision composition |
| Low-$z$ | $0.01<z<0.30$ | redshift-domain comparison |
| High-$z$ | $z\ge0.30$ | redshift-domain comparison |
| Omit SDSS | `IDSURVEY != 1` | survey composition / influence |

The thresholds are prespecified so that submissions answer comparable statistical  
questions.

## E1. Defining “largest uncertainty”

For the baseline full covariance, define the individual diagonal scale

$$
\sigma_i
=
\sqrt{C_{ii}}.
$$

The top-10% perturbation removes rows above the baseline 90th percentile of this  
quantity.

This criterion uses the covariance diagonal only to **define the subset**.

The fit itself still uses the correctly sliced **full covariance matrix**.

In [ ]:
z0 = data0[
    "zHD"
].to_numpy(float)

sigma0 = np.sqrt(
    np.diag(C0)
)

sigma90 = np.quantile(
    sigma0,
    0.90,
)

variant_masks = {
    "Baseline": np.ones(
        len(data0),
        dtype=bool,
    ),
    "z <= 0.70": (
        z0 <= 0.70
    ),
    "Remove top 10% sigma": (
        sigma0 <= sigma90
    ),
    "Low z (< 0.30)": (
        z0 < 0.30
    ),
    "High z (>= 0.30)": (
        z0 >= 0.30
    ),
    "Omit SDSS": (
        data0[
            "IDSURVEY"
        ].to_numpy(int)
        != 1
    ),
}

for name, mask in variant_masks.items():
    print(
        f"{name:22s}",
        int(mask.sum()),
    )

## E2. Safe subsetting within the baseline sample

The masks above are defined relative to the baseline rows.

For a variant mask $J$,

$$
D_J
$$

and

$$
C_{J,J}
$$

must be passed together to the same posterior engine.

In [ ]:
def subset_baseline(
    data,
    covariance,
    mask,
):
    idx = np.flatnonzero(mask)

    d = (
        data.iloc[idx]
        .reset_index(drop=True)
        .copy()
    )

    c = covariance[
        np.ix_(idx, idx)
    ]

    assert (
        len(d)
        == c.shape[0]
        == c.shape[1]
    )

    return d, c

variant_data = {}

for name, mask in variant_masks.items():
    d, c = subset_baseline(
        data0,
        C0,
        mask,
    )

    variant_data[name] = (
        d,
        c,
    )

# F. Audit sample composition before fitting

Before looking at cosmological results, inspect how each perturbation changes the  
observed sample.

For every variant record:

- number of rows $N$;
- number of unique physical `CID`;
- median redshift;
- maximum redshift;
- median diagonal uncertainty;
- number of represented surveys.

This prevents a later parameter shift from being interpreted without knowing what  
changed in the data.

In [ ]:
audit_rows = []

for name, (d, c) in variant_data.items():
    audit_rows.append({
        "variant": name,
        "N": len(d),
        "unique_CID": d[
            "CID"
        ].nunique(),
        "median_z": float(
            d["zHD"].median()
        ),
        "max_z": float(
            d["zHD"].max()
        ),
        "median_sigma": float(
            np.median(
                np.sqrt(
                    np.diag(c)
                )
            )
        ),
        "N_surveys": int(
            d[
                "IDSURVEY"
            ].nunique()
        ),
    })

audit_table = pd.DataFrame(
    audit_rows
)

display(
    audit_table.round(5)
)

> ### Exercise 1 — Before fitting
> In 100–150 words, predict which compulsory perturbation you expect to move
> $\Omega_m$ most and why.
>
> Your answer must distinguish **redshift leverage**, **precision** and
> **survey composition**. This is a prediction, not a graded requirement that
> the data behave as you expect.

# G. Fit every fixed variant

Use the same posterior engine for every sample.

For variant $k$, define

$$
\Delta\Omega_{m,k}
=
\Omega_{m,50}^{(k)}
-
\Omega_{m,50}^{(0)}
$$

and

$$
\Delta M_{B,k}
=
M_{B,50}^{(k)}
-
M_{B,50}^{(0)}.
$$

Also compute the descriptive normalized shift

$$
R_k
=
\frac{
|\Delta\Omega_{m,k}|
}{
(\Omega_{m,84}^{(0)}
-
\Omega_{m,16}^{(0)})/2
}.
$$

**Important:** $R_k$ is a descriptive scale, not a formal tension significance. The  
perturbed and baseline samples overlap and are statistically dependent.

In [ ]:
fits = {}

for name, (d, c) in variant_data.items():
    print("Fitting:", name)

    fits[name] = fit_variant(
        d,
        c,
    )

In [ ]:
baseline_half_width = (
    baseline_om[2]
    - baseline_om[0]
) / 2.0

result_rows = []

for name, fit in fits.items():
    om16, om50, om84 = fit[
        "q_omega"
    ]

    mb16, mb50, mb84 = fit[
        "q_mb"
    ]

    delta_om = (
        om50
        - baseline_om[1]
    )

    delta_mb = (
        mb50
        - baseline_mb[1]
    )

    result_rows.append({
        "variant": name,
        "N": len(
            variant_data[name][0]
        ),
        "Omega_m_median": om50,
        "Omega_m_16": om16,
        "Omega_m_84": om84,
        "Omega_m_width": (
            om84 - om16
        ),
        "Delta_Omega_m": delta_om,
        "R_abs_shift": (
            abs(delta_om)
            / baseline_half_width
        ),
        "M_B_median": mb50,
        "M_B_16": mb16,
        "M_B_84": mb84,
        "M_B_width": (
            mb84 - mb16
        ),
        "Delta_M_B": delta_mb,
        "edge_mass": fit[
            "edge_mass"
        ],
    })

results = pd.DataFrame(
    result_rows
)

display(
    results.round(6)
)

## Exercise 2 — Interpret the robustness table

Answer all four.

1. Which perturbation gives the largest
$|\Delta\Omega_m|$?
2. Does that perturbation mainly shift the posterior location, widen the
posterior, or both?
3. Which perturbation most changes the $M_B$ posterior?
4. Do any posteriors approach a prior/grid boundary?

Do not use overlap of 68% intervals as a formal hypothesis test.

# H. Robustness plots

Create a forest-style plot for $\Omega_m$ and a separate one for $M_B$.

For each row:

- point = posterior median;
- horizontal interval = central 68% interval;
- vertical reference line = baseline median.

These figures summarize **sensitivity**, not selection-corrected cosmology.

In [ ]:
plot_table = results.copy()

y = np.arange(
    len(plot_table)
)

fig, ax = plt.subplots(
    figsize=(8, 5.5)
)

x = plot_table[
    "Omega_m_median"
].to_numpy()

xerr = np.vstack([
    x
    - plot_table[
        "Omega_m_16"
    ].to_numpy(),
    plot_table[
        "Omega_m_84"
    ].to_numpy()
    - x,
])

ax.errorbar(
    x,
    y,
    xerr=xerr,
    fmt="o",
    capsize=3,
)

ax.axvline(
    baseline_om[1],
    linestyle="--",
)

ax.set_yticks(
    y,
    plot_table[
        "variant"
    ],
)

ax.set_xlabel(
    r"$\Omega_m$"
)

ax.set_title(
    "Robustness of the matter-density posterior"
)

ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 5.5)
)

x = plot_table[
    "M_B_median"
].to_numpy()

xerr = np.vstack([
    x
    - plot_table[
        "M_B_16"
    ].to_numpy(),
    plot_table[
        "M_B_84"
    ].to_numpy()
    - x,
])

ax.errorbar(
    x,
    y,
    xerr=xerr,
    fmt="o",
    capsize=3,
)

ax.axvline(
    baseline_mb[1],
    linestyle="--",
)

ax.set_yticks(
    y,
    plot_table[
        "variant"
    ],
)

ax.set_xlabel(
    r"$M_B$ [mag]"
)

ax.set_title(
    "Robustness of the nuisance-magnitude posterior"
)

ax.invert_yaxis()

plt.tight_layout()
plt.show()

## In-class checkpoint

Before moving to the graded extension, show:

1. completed five-line scientific statement;
2. reproduced Lab 05 baseline;
3. sample-audit table;
4. full robustness results table;
5. $\Omega_m$ forest plot;
6. $M_B$ forest plot;
7. one-sentence identification of the largest tested
$|\Delta\Omega_m|$.

A plot without the numerical table is not a complete checkpoint.

# I. Diagnose the largest shift

A large shift is a **diagnostic result**, not an instruction to delete the  
corresponding observations.

Identify the non-baseline variant with the largest

$$
|\Delta\Omega_m|.
$$

Then compare that sample with the baseline in at least three ways:

1. redshift distribution;
2. diagonal uncertainty distribution $\sqrt{C_{ii}}$;
3. survey composition.

If you want to use the word **outlier**, you must additionally examine a residual or  
influence diagnostic. A sample shift alone does not establish that any observation is  
an outlier.

In [ ]:
nonbaseline = results[
    results["variant"]
    != "Baseline"
].copy()

largest_name = (
    nonbaseline
    .loc[
        nonbaseline[
            "Delta_Omega_m"
        ].abs().idxmax(),
        "variant",
    ]
)

print(
    "Largest absolute Omega_m shift:",
    largest_name,
)

largest_data, largest_cov = (
    variant_data[
        largest_name
    ]
)

comparison = pd.DataFrame({
    "quantity": [
        "N",
        "median z",
        "90th percentile z",
        "median sigma",
        "90th percentile sigma",
        "number of surveys",
    ],
    "baseline": [
        len(data0),
        data0["zHD"].median(),
        data0["zHD"].quantile(0.90),
        np.median(
            np.sqrt(
                np.diag(C0)
            )
        ),
        np.quantile(
            np.sqrt(
                np.diag(C0)
            ),
            0.90,
        ),
        data0["IDSURVEY"].nunique(),
    ],
    "largest-shift variant": [
        len(largest_data),
        largest_data["zHD"].median(),
        largest_data["zHD"].quantile(0.90),
        np.median(
            np.sqrt(
                np.diag(largest_cov)
            )
        ),
        np.quantile(
            np.sqrt(
                np.diag(largest_cov)
            ),
            0.90,
        ),
        largest_data[
            "IDSURVEY"
        ].nunique(),
    ],
})

display(
    comparison.round(5)
)

## Exercise 3 — Largest-shift diagnosis

Produce **two diagnostic figures** and a 150–250 word interpretation.

At least one figure must address redshift or uncertainty distribution.

If the largest-shift variant is the survey omission, also show how the survey removal  
changes the redshift distribution.

Your interpretation must distinguish:

- what the data show;
- a plausible mechanism;
- what additional analysis would be needed to establish that mechanism.

# J. Claims discipline

Use the table below when writing your conclusion.

| Statement | Supported by Lab 06? |
|---|---|
| “The result is stable under the tested perturbations.” | **Possibly**, if the numerical results support it |
| “Variant X gives the largest tested shift.” | **Yes**, if demonstrated |
| “The high-$z$ tail has strong leverage.” | **Possibly**, if diagnosed |
| “SDSS is wrong.” | **No** |
| “The observed sample is unbiased.” | **No** |
| “Malmquist bias has been removed.” | **No** |
| “The Pantheon+ selection function has been corrected.” | **No** |

Survey omission can reveal sensitivity to survey composition. It cannot by itself  
identify whether the mechanism is calibration, redshift coverage, precision, sample  
size or targeting.

# K. Extended Analysis — graded

Complete **all** tasks below independently.

The code for the baseline posterior and fixed perturbations is supplied so that the  
grade emphasizes scientific reasoning, statistical validity, robustness and  
reproducibility rather than boilerplate.

## E1 — Quantitative robustness table

Submit the complete table for all six compulsory sample definitions.

For each variant include:

- $N$;
- $\Omega_m$ median and central 68% interval;
- $\Omega_m$ interval width;
- $\Delta\Omega_m$ from baseline;
- descriptive normalized shift $R$;
- $M_B$ median and central 68% interval;
- $\Delta M_B$ from baseline.

Include **two quantitative sentences** describing the largest changes.

Do not call $R$ a “number of sigma tension”.

## E2 — Largest-shift diagnosis

Identify the largest

$$
|\Delta\Omega_m|.
$$

Compare the corresponding sample with baseline using:

- redshift distribution;
- diagonal uncertainty distribution;
- survey composition.

Explain whether the evidence points most strongly toward redshift leverage, precision  
composition, survey composition, or a mixture.

If the mechanism is not identifiable from these diagnostics, say so.

## E3 — Influence versus invalid data

Answer in 150–250 words.

1. Why does a large parameter shift after removing a subset imply
**influence/sensitivity**, not automatically invalid data?
2. What additional diagnostic would you require before describing individual
observations as outliers?
3. If omitting SDSS causes a shift, why could survey and redshift effects be
confounded?

## E4 — Sensitivity versus formal selection correction

Starting from

$$
p(D\mid\theta,S=1)
=
\frac{
p(S=1\mid D,\theta)\,p(D\mid\theta)
}{
P(S=1\mid\theta)
},
$$

explain which quantities required for a formal selection correction are **not  
estimated in this lab**.

Then explain precisely what the sample perturbations *do* establish.

## E5 — Scientific conclusion (300–500 words)

In connected scientific prose answer:

- What was inferred?
- What baseline sample/model/covariance was conditioned upon?
- Which perturbation produced the largest tested shift?
- How large was that shift?
- Which aspect of the result was most stable?
- What diagnostic evidence helps explain the largest shift?
- What is the dominant limitation of this robustness exercise?
- Why is the result a sensitivity statement rather than a selection-function
correction?
- What stronger assertion would be unjustified?

### Required final sentence

> **“From this lab we can conclude ___, conditional on ___, but we cannot
> conclude ___.”**

## Student workspace for E1–E4

Add your code, tables, figures and explanations below. Label all axes and state units  
where relevant.

In [ ]:
# E1–E4 student implementation.
# Add additional code and markdown cells as needed.

## E5 — Scientific conclusion

[Replace this text with your 300–500 word conclusion.]

# Submission and grading checklist

Before submission:

- restart the kernel;
- run all cells from top to bottom;
- resolve every incomplete exercise;
- keep the official data provenance visible;
- export the same executed notebook as HTML/PDF;
- submit `Surname_Name_L06.ipynb` plus that export.

| Dimension | Weight | Evidence expected |
|---|---:|---|
| Scientific formulation | 20% | Specific estimand, model, covariance, sample conditioning and limitation |
| Statistical correctness | 25% | Correct masks, covariance slicing, fixed posterior engine and shift definitions |
| Uncertainty quantification | 20% | 68% intervals, widths, absolute shifts and careful interpretation of $R$ |
| Diagnostics / failure analysis | 15% | Baseline validation, largest-shift diagnosis, influence/selection reasoning |
| Reproducibility | 10% | Prespecified perturbations, official data, full execution, readable tables/plots |
| Scientific interpretation | 10% | 300–500 word conclusion with supported and unsupported claims separated |

### Scientific scope

This notebook is a teaching robustness analysis. It is not an official Pantheon+  
cosmology reproduction, an independent $H_0$ measurement, or a formal  
selection-function correction.